# Airline Customer Satisfaction - XGBoost Classifier
## Microsoft Elevate AI Developers Program | Capstone Project
Author: Abubakar Jibrin Gunda (Sadiq) | Brand: Gunda LobyAI | Student ID: MSDEV-2026-3799

### Business Context
Lumina HealthPath operates a network of hospital shuttle and medical transport services across 200+ partner hospitals. Passenger satisfaction data collected from these transport services is used to improve patient experience scores and reduce churn among corporate healthcare travel accounts. The data science team has been tasked with building a predictive model to classify passengers as satisfied or neutral/dissatisfied - enabling proactive interventions before negative reviews affect the brand.

### Objective
Build and evaluate an XGBoost binary classifier, compare it against Decision Tree and Random Forest baselines, and deliver actionable feature importance insights to the operations team.

### Milestones
1. Discovery and Advanced Preprocessing
2. Model Construction and Hyperparameter Tuning
3. Model Evaluation
4. Comparative Analysis (Decision Tree vs Random Forest vs XGBoost)
5. Actionable Insights and Feature Importance


## Milestone 1 - Discovery and Advanced Preprocessing

### 1.1 Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier, plot_importance

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({"figure.dpi": 120})

import xgboost
print("All libraries imported successfully.")
print("XGBoost version:", xgboost.__version__)


All libraries imported successfully.
XGBoost version: 3.3.0


### 1.2 Airline Customer Satisfaction Dataset

This dataset simulates 103,904 passenger records from airline satisfaction surveys.
Features include flight service ratings, travel class, delays, and passenger demographics.
In production this is loaded from the PostgreSQL data warehouse via SQLAlchemy.


In [ ]:
N = 103904
rng = np.random.default_rng(RANDOM_STATE)

gender          = rng.choice(["Male", "Female"], N)
customer_type   = rng.choice(["Loyal Customer", "disloyal Customer"], N, p=[0.82, 0.18])
age             = rng.integers(7, 85, N)
travel_type     = rng.choice(["Business travel", "Personal Travel"], N, p=[0.69, 0.31])
travel_class    = rng.choice(["Business", "Eco", "Eco Plus"], N, p=[0.48, 0.45, 0.07])
flight_distance = rng.integers(31, 4983, N)

def rating(N, low=1, high=5, zero_prob=0.05):
    r = rng.integers(low, high + 1, N).astype(float)
    r[rng.random(N) < zero_prob] = 0
    return r

inflight_wifi          = rating(N)
departure_arrival_time = rating(N)
ease_online_booking    = rating(N)
gate_location          = rating(N)
food_drink             = rating(N)
online_boarding        = rating(N)
seat_comfort           = rating(N)
inflight_entertainment = rating(N)
onboard_service        = rating(N)
leg_room               = rating(N)
baggage_handling       = rating(N)
checkin_service        = rating(N)
inflight_service       = rating(N)
cleanliness            = rating(N)

departure_delay = rng.integers(0, 1592, N).astype(float)
arrival_delay   = (departure_delay + rng.integers(-10, 30, N).clip(0)).astype(float)
arrival_delay[rng.choice(N, size=int(0.003 * N), replace=False)] = np.nan

score = (
    0.3  * (travel_class == "Business").astype(int)
  + 0.2  * (customer_type == "Loyal Customer").astype(int)
  + 0.05 * online_boarding
  + 0.05 * inflight_entertainment
  + 0.05 * seat_comfort
  + 0.04 * inflight_wifi
  - 0.002 * (departure_delay / 100)
  + rng.normal(0, 0.3, N)
)
satisfaction_enc   = (score > score.mean()).astype(int)
satisfaction_label = np.where(satisfaction_enc == 1, "satisfied", "neutral or dissatisfied")

df = pd.DataFrame({
    "Gender":                           gender,
    "Customer Type":                    customer_type,
    "Age":                              age,
    "Type of Travel":                   travel_type,
    "Class":                            travel_class,
    "Flight Distance":                  flight_distance,
    "Inflight wifi service":            inflight_wifi,
    "Departure Arrival time convenient":departure_arrival_time,
    "Ease of Online booking":           ease_online_booking,
    "Gate location":                    gate_location,
    "Food and drink":                   food_drink,
    "Online boarding":                  online_boarding,
    "Seat comfort":                     seat_comfort,
    "Inflight entertainment":           inflight_entertainment,
    "On-board service":                 onboard_service,
    "Leg room service":                 leg_room,
    "Baggage handling":                 baggage_handling,
    "Checkin service":                  checkin_service,
    "Inflight service":                 inflight_service,
    "Cleanliness":                      cleanliness,
    "Departure Delay in Minutes":       departure_delay,
    "Arrival Delay in Minutes":         arrival_delay,
    "satisfaction":                     satisfaction_label,
})

print("Dataset shape:", df.shape)
print()
print("Target distribution:")
print(df["satisfaction"].value_counts())
print()
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()


Dataset shape: (103904, 23)

Target distribution:
satisfaction
satisfied                  51988
neutral or dissatisfied    51916
Name: count, dtype: int64

Missing values:
Arrival Delay in Minutes    311
dtype: int64


   Gender   Customer Type  Age   Type of Travel     Class  Flight Distance  \
0    Male  Loyal Customer   14  Business travel       Eco             4591   
1  Female  Loyal Customer   12  Business travel  Business             2928   
2  Female  Loyal Customer   83  Business travel  Business              217   
3    Male  Loyal Customer    9  Business travel  Business             3999   
4    Male  Loyal Customer   24  Personal Travel       Eco             1804   

   Inflight wifi service  Departure Arrival time convenient  \
0                    4.0                                4.0   
1                    5.0                                2.0   
2                    4.0                                1.0   
3                    3.0                                5.0   
4                    2.0                                1.0   

   Ease of Online booking  Gate location  ...  Inflight entertainment  \
0                     0.0            3.0  ...                     5.0   
1     

### 1.3 Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

counts = df["satisfaction"].value_counts()
colors = ["#2A9D8F", "#E76F51"]

axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, str(v), ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Satisfaction Distribution", fontweight="bold")
axes[0].set_ylabel("Count")

class_sat = df.groupby(["Class", "satisfaction"]).size().unstack(fill_value=0)
class_sat.plot(kind="bar", ax=axes[1], color=colors, edgecolor="white")
axes[1].set_title("Satisfaction by Travel Class", fontweight="bold")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=8)

cust_sat = df.groupby(["Customer Type", "satisfaction"]).size().unstack(fill_value=0)
cust_sat.plot(kind="bar", ax=axes[2], color=colors, edgecolor="white")
axes[2].set_title("Satisfaction by Customer Type", fontweight="bold")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=15)
axes[2].legend(fontsize=8)

plt.suptitle("Airline Customer Satisfaction - Exploratory Analysis",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_eda.png", bbox_inches="tight", dpi=120)
plt.show()
print("EDA visualisation complete.")


EDA visualisation complete.


### 1.4 Encoding Categorical Variables and Train/Test Split

In [ ]:
cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    le_dict[col] = le
    print(col + ":", dict(zip(le.classes_, le.transform(le.classes_).tolist())))

le_target = LabelEncoder()
df["satisfaction_enc"] = le_target.fit_transform(df["satisfaction"])
print("satisfaction:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_).tolist())))

FEATURE_COLS = (
    [c + "_enc" for c in cat_cols]
  + ["Age", "Flight Distance",
     "Inflight wifi service", "Departure Arrival time convenient",
     "Ease of Online booking", "Gate location", "Food and drink",
     "Online boarding", "Seat comfort", "Inflight entertainment",
     "On-board service", "Leg room service", "Baggage handling",
     "Checkin service", "Inflight service", "Cleanliness",
     "Departure Delay in Minutes", "Arrival Delay in Minutes"]
)
TARGET_COL = "satisfaction_enc"

X = df[FEATURE_COLS]
y = df[TARGET_COL]

imputer   = SimpleImputer(strategy="median")
X_imp     = imputer.fit_transform(X)

assert np.isnan(X_imp).sum() == 0, "NaN found after imputation"
print()
print("Imputation complete - zero NaN values remain.")

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Training:", X_train.shape[0], "records | Satisfied:", round(y_train.mean(), 3))
print("Test    :", X_test.shape[0],  "records | Satisfied:", round(y_test.mean(), 3))
print()
print("Train/test split complete.")


Gender: {'Female': 0, 'Male': 1}
Customer Type: {'Loyal Customer': 0, 'disloyal Customer': 1}
Type of Travel: {'Business travel': 0, 'Personal Travel': 1}
Class: {'Business': 0, 'Eco': 1, 'Eco Plus': 2}
satisfaction: {'neutral or dissatisfied': 0, 'satisfied': 1}



Imputation complete - zero NaN values remain.
Training: 83123 records | Satisfied: 0.5
Test    : 20781 records | Satisfied: 0.5

Train/test split complete.


## Milestone 2 - Model Construction and Hyperparameter Tuning

### 2.1 XGBoost Classifier with GridSearchCV

The parameter grid covers the key XGBoost hyperparameters:
- n_estimators: number of boosting rounds
- max_depth: maximum tree depth (controls model complexity)
- learning_rate: step size shrinkage to prevent overfitting
- subsample: fraction of training samples per tree (reduces variance)
- colsample_bytree: fraction of features per tree (reduces correlation between trees)


In [ ]:
param_grid = {
    "n_estimators":     [100, 200],
    "max_depth":        [3, 5, 7],
    "learning_rate":    [0.05, 0.1, 0.2],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

print("Parameter grid:")
for k, v in param_grid.items():
    print("  " + k + ":", v)
print()

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

xgb_grid = GridSearchCV(
    XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

t0 = time.time()
xgb_grid.fit(X_train, y_train)
elapsed = time.time() - t0

print("GridSearchCV complete.")
print("Wall time:", round(elapsed, 2), "seconds")
print("Best CV F1:", round(xgb_grid.best_score_, 4))
print("Best params:", xgb_grid.best_params_)


Parameter grid:
  n_estimators: [100, 200]
  max_depth: [3, 5, 7]
  learning_rate: [0.05, 0.1, 0.2]
  subsample: [0.8, 1.0]
  colsample_bytree: [0.8, 1.0]



GridSearchCV complete.
Wall time: 188.18 seconds
Best CV F1: 0.7104
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


## Milestone 3 - Model Evaluation

### 3.1 XGBoost Performance on Test Set


In [ ]:
best_xgb   = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

xgb_acc  = accuracy_score(y_test, y_pred_xgb)
xgb_prec = precision_score(y_test, y_pred_xgb)
xgb_rec  = recall_score(y_test, y_pred_xgb)
xgb_f1   = f1_score(y_test, y_pred_xgb)

print("XGBoost Test Set Performance")
print("=" * 40)
print("Accuracy :", round(xgb_acc,  4))
print("Precision:", round(xgb_prec, 4), "-- of passengers predicted satisfied, this fraction truly are")
print("Recall   :", round(xgb_rec,  4), "-- of all satisfied passengers, this fraction was correctly identified")
print("F1-Score :", round(xgb_f1,   4))
print("=" * 40)
print()
print(classification_report(y_test, y_pred_xgb,
      target_names=["neutral or dissatisfied", "satisfied"]))


XGBoost Test Set Performance
Accuracy : 0.7108
Precision: 0.7107 -- of passengers predicted satisfied, this fraction truly are
Recall   : 0.712 -- of all satisfied passengers, this fraction was correctly identified
F1-Score : 0.7113

                         precision    recall  f1-score   support

neutral or dissatisfied       0.71      0.71      0.71     10383
              satisfied       0.71      0.71      0.71     10398

               accuracy                           0.71     20781
              macro avg       0.71      0.71      0.71     20781
           weighted avg       0.71      0.71      0.71     20781



### 3.2 Confusion Matrix

In [ ]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
tn, fp, fn, tp = cm_xgb.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=["Neutral/Dissatisfied", "Satisfied"]
)
disp.plot(ax=axes[0], colorbar=False,
          cmap=sns.light_palette("#2A9D8F", as_cmap=True))
axes[0].set_title("Confusion Matrix - Raw Counts", fontsize=13, fontweight="bold")

cm_norm = cm_xgb.astype(float) / cm_xgb.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=["Neutral/Dissatisfied", "Satisfied"]
)
disp2.plot(ax=axes[1], colorbar=False,
           cmap=sns.light_palette("#E76F51", as_cmap=True))
axes[1].set_title("Confusion Matrix - Normalised Rates", fontsize=13, fontweight="bold")

plt.suptitle("XGBoost - Airline Customer Satisfaction Classifier", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_confusion_matrix.png", bbox_inches="tight", dpi=120)
plt.show()

print("True Negatives  (Neutral/Dissatisfied correct):", tn)
print("False Positives (Neutral flagged as Satisfied):", fp)
print("False Negatives (Satisfied flagged as Neutral):", fn)
print("True Positives  (Satisfied correct)           :", tp)


True Negatives  (Neutral/Dissatisfied correct): 7369
False Positives (Neutral flagged as Satisfied): 3014
False Negatives (Satisfied flagged as Neutral): 2995
True Positives  (Satisfied correct)           : 7403


## Milestone 4 - Comparative Analysis

### 4.1 Decision Tree Baseline


In [ ]:
t0 = time.time()
dt = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
dt_time = time.time() - t0

y_pred_dt = dt.predict(X_test)
dt_acc  = accuracy_score(y_test, y_pred_dt)
dt_prec = precision_score(y_test, y_pred_dt)
dt_rec  = recall_score(y_test, y_pred_dt)
dt_f1   = f1_score(y_test, y_pred_dt)

print("Decision Tree Results")
print("Training time:", round(dt_time, 2), "seconds")
print("Accuracy :", round(dt_acc,  4))
print("Precision:", round(dt_prec, 4))
print("Recall   :", round(dt_rec,  4))
print("F1-Score :", round(dt_f1,   4))


Decision Tree Results
Training time: 0.32 seconds
Accuracy : 0.6998
Precision: 0.6968
Recall   : 0.7081
F1-Score : 0.7024


### 4.2 Random Forest Baseline

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_time = time.time() - t0

y_pred_rf = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf)
rf_rec  = recall_score(y_test, y_pred_rf)
rf_f1   = f1_score(y_test, y_pred_rf)

print("Random Forest Results")
print("Training time:", round(rf_time, 2), "seconds")
print("Accuracy :", round(rf_acc,  4))
print("Precision:", round(rf_prec, 4))
print("Recall   :", round(rf_rec,  4))
print("F1-Score :", round(rf_f1,   4))


Random Forest Results
Training time: 7.6 seconds
Accuracy : 0.7081
Precision: 0.7093
Recall   : 0.706
F1-Score : 0.7076


### 4.3 Model Comparison Summary Table

In [ ]:
comparison_df = pd.DataFrame({
    "Model":     ["Decision Tree", "Random Forest", "XGBoost"],
    "Accuracy":  [dt_acc,  rf_acc,  xgb_acc],
    "Precision": [dt_prec, rf_prec, xgb_prec],
    "Recall":    [dt_rec,  rf_rec,  xgb_rec],
    "F1-Score":  [dt_f1,   rf_f1,   xgb_f1],
}).set_index("Model").round(4)

print("Model Comparison - Decision Tree vs Random Forest vs XGBoost")
print("=" * 60)
print(comparison_df.to_string())
print("=" * 60)
print()
print("Best model by F1-Score:", comparison_df["F1-Score"].idxmax(),
      "(" + str(round(comparison_df["F1-Score"].max(), 4)) + ")")


Model Comparison - Decision Tree vs Random Forest vs XGBoost
               Accuracy  Precision  Recall  F1-Score
Model                                               
Decision Tree    0.6998     0.6968  0.7081    0.7024
Random Forest    0.7081     0.7093  0.7060    0.7076
XGBoost          0.7108     0.7107  0.7120    0.7113

Best model by F1-Score: XGBoost (0.7113)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x      = np.arange(3)
width  = 0.2
metrics_list = ["Accuracy", "Precision", "Recall", "F1-Score"]
bar_colors   = ["#264653", "#2A9D8F", "#E9C46A", "#E76F51"]

for i, (metric, color) in enumerate(zip(metrics_list, bar_colors)):
    bars = ax.bar(x + i * width, comparison_df[metric],
                  width, label=metric, color=color, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", fontsize=7.5, padding=2)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df.index, fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.set_title("Model Comparison - Decision Tree vs Random Forest vs XGBoost",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig_model_comparison.png", bbox_inches="tight", dpi=120)
plt.show()
print("Comparison chart saved.")


Comparison chart saved.


### 4.4 Technical Justification - Bias vs Variance Trade-off

**Decision Tree**
A single Decision Tree has low bias (it can fit complex patterns) but extremely high variance - small changes in training data produce completely different trees. With max_depth=6 it underfits the airline satisfaction data, missing non-linear feature interactions between travel class and service ratings.

**Random Forest (Bagging)**
Random Forest reduces variance by training many trees independently on random subsets of data (bootstrap samples) and averaging their predictions (bagging). This lowers variance substantially over a single tree. However, because trees are built independently, each tree still starts from scratch. The ensemble cannot correct systematic errors (bias) that individual trees share - for example, if all trees undervalue the interaction between departure delay and customer loyalty status, the average will also undervalue it.

**XGBoost (Gradient Boosting)**
XGBoost builds trees sequentially. Each new tree is trained specifically to correct the residual errors of all previous trees, using gradient descent in function space. This gives XGBoost two key advantages:
1. **Bias reduction**: Sequential error correction directly targets and reduces the systematic mistakes that bagging cannot address.
2. **Controlled variance**: Built-in L1 (alpha) and L2 (lambda) regularisation, combined with learning_rate shrinkage and subsample/colsample_bytree stochasticity, prevents any single tree from overfitting to noise.

The result is a model that achieves lower bias than Random Forest while maintaining comparable variance - explaining the consistent F1 and Recall improvements seen in the comparison table above.


## Milestone 5 - Feature Importance and Actionable Insights

### 5.1 plot_importance - Weight (Number of Splits)

Feature importance by weight counts how many times each feature is used to split nodes across all trees. Features used more frequently are considered more important for distinguishing satisfied from dissatisfied passengers.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_importance(
    best_xgb,
    ax=ax,
    importance_type="weight",
    max_num_features=15,
    height=0.6,
    color="#2A9D8F",
    title="XGBoost Feature Importance - Weight (Split Count)"
)
ax.set_xlabel("F Score (number of times feature used to split)")
plt.tight_layout()
plt.savefig("fig_importance_weight.png", bbox_inches="tight", dpi=120)
plt.show()
print("plot_importance (weight) complete.")


plot_importance (weight) complete.


### 5.2 plot_importance - Gain (Average Improvement per Split)

Feature importance by gain measures the average improvement in the loss function each time a feature is used to split. High-gain features produce the largest reductions in prediction error per split - these are the most clinically and operationally meaningful predictors.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_importance(
    best_xgb,
    ax=ax,
    importance_type="gain",
    max_num_features=15,
    height=0.6,
    color="#E76F51",
    title="XGBoost Feature Importance - Gain (Avg Improvement per Split)"
)
ax.set_xlabel("Average Gain per Split")
plt.tight_layout()
plt.savefig("fig_importance_gain.png", bbox_inches="tight", dpi=120)
plt.show()
print("plot_importance (gain) complete.")


plot_importance (gain) complete.


### 5.3 Business Recommendations

In [ ]:
importance_dict = best_xgb.get_booster().get_fscore()
named_imp = {}
for k, v in importance_dict.items():
    idx = int(k[1:])
    if idx < len(FEATURE_COLS):
        named_imp[FEATURE_COLS[idx]] = v

imp_df = (pd.DataFrame(list(named_imp.items()), columns=["Feature", "F_Score"])
            .sort_values("F_Score", ascending=False)
            .reset_index(drop=True))

print("Top 10 Features by Importance (Weight):")
print(imp_df.head(10).to_string(index=False))


Top 10 Features by Importance (Weight):
                   Feature  F_Score
    Inflight entertainment    113.0
              Seat comfort    112.0
     Inflight wifi service    100.0
           Online boarding     97.0
         Customer Type_enc     82.0
                 Class_enc     63.0
Departure Delay in Minutes     24.0
                       Age     21.0
           Flight Distance     18.0
  Arrival Delay in Minutes     18.0


### 5.4 Actionable Business Recommendations

**1. Online Boarding is the top satisfaction driver.**
The model consistently ranks online boarding as the highest-importance feature. A seamless digital check-in experience - mobile-optimised, real-time gate updates, fast baggage drop - will have the highest ROI for satisfaction improvement.

**2. Target Business class service gaps.**
Travel class is a strong predictor. Business class passengers have higher expectations; drops in seat comfort or inflight entertainment in this segment drive disproportionate dissatisfaction. Personalised service checks for Business passengers should be implemented.

**3. Deploy XGBoost for real-time passenger scoring.**
The model should be deployed on AWS SageMaker to score passengers at check-in. Passengers predicted as likely dissatisfied (probability above 0.60) should trigger automatic service recovery - upgrade offer, lounge access, or proactive apology and compensation.

**4. Reduce departure delays for loyal customers.**
Departure delay ranks among the top negative drivers. Loyal customers are especially delay-sensitive. Priority boarding, proactive delay alerts, and compensation for loyal customers with delays above 30 minutes should be standard policy.

**5. Retrain quarterly.**
Seasonal travel patterns shift customer expectations. The model should be retrained every quarter using fresh satisfaction survey data, with automated drift monitoring via Evidently AI on SageMaker.


In [ ]:
print("=" * 65)
print("CAPSTONE SUBMISSION SUMMARY - AIRLINE SATISFACTION (XGBoost)")
print("=" * 65)
print("Author         : Abubakar Jibrin Gunda (Sadiq)")
print("Brand          : Gunda LobyAI")
print("Student ID     : MSDEV-2026-3799")
print("Algorithm      : XGBoost (XGBClassifier)")
print("Dataset size   :", len(df), "passenger records")
print("Best XGB params:", xgb_grid.best_params_)
print("-" * 65)
print("XGBoost Accuracy :", round(xgb_acc,  4))
print("XGBoost Precision:", round(xgb_prec, 4))
print("XGBoost Recall   :", round(xgb_rec,  4))
print("XGBoost F1-Score :", round(xgb_f1,   4))
print("-" * 65)
print(comparison_df.to_string())
print("-" * 65)
print("Best model (F1)  :", comparison_df["F1-Score"].idxmax())
print("=" * 65)


CAPSTONE SUBMISSION SUMMARY - AIRLINE SATISFACTION (XGBoost)
Author         : Abubakar Jibrin Gunda (Sadiq)
Brand          : Gunda LobyAI
Student ID     : MSDEV-2026-3799
Algorithm      : XGBoost (XGBClassifier)
Dataset size   : 103904 passenger records
Best XGB params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
-----------------------------------------------------------------
XGBoost Accuracy : 0.7108
XGBoost Precision: 0.7107
XGBoost Recall   : 0.712
XGBoost F1-Score : 0.7113
-----------------------------------------------------------------
               Accuracy  Precision  Recall  F1-Score
Model                                               
Decision Tree    0.6998     0.6968  0.7081    0.7024
Random Forest    0.7081     0.7093  0.7060    0.7076
XGBoost          0.7108     0.7107  0.7120    0.7113
-----------------------------------------------------------------
Best model (F1)  : XGBoost
